# Jacobian 引导调度的形式化权衡证书

这个 notebook 单独抽出调度实验背后的数学命题。实验检验的是局部模型什么时候足够准确；下面的定理证明的是，一旦接受局部模型给出的测量量，哪些结论会被数学上推出。

## 设定

执行某个候选调度的短前缀后，假设我们测得

$$
L(a)>0,
\qquad
\lambda(a).
$$

这里 $L(a)$ 是前缀后的 loss，$\lambda(a)$ 是在到达状态处由 Jacobian 计算得到的局部有效收缩率。对剩余 horizon $H$、学习率 $\eta$ 和折扣 $\alpha$，定义

$$
c=2\alpha\eta H,
\qquad c>0.
$$

局部价值模型预测

$$
\widehat L(a)=L(a)\exp\{-c\lambda(a)\}.
$$

## 主定理：权衡证书

对两个候选 $a$ 和 $b$，只要

$$
\log\frac{L(b)}{L(a)}
<
c\bigl(\lambda(b)-\lambda(a)\bigr),
$$

就可以预测 $b$ 优于 $a$。也就是说，上式推出

$$
L(b)\exp\{-c\lambda(b)\}
<
L(a)\exp\{-c\lambda(a)\}.
$$

换句话说：一个调度的前缀 loss 可以更差，但只要它的未来收缩优势足以偿还这个对数形式的前缀 loss 代价，它就会被预测为更好。

## 推论 1：正的局部速率给出一阶下降

黑盒 Jacobian 小节使用一阶近似

$$
\|e_{t+1}\|^2
\approx
\|e_t\|^2(1-2\rho_t).
$$

其代数核心是

$$
\rho_t>0
\quad\Longrightarrow\quad
1-2\rho_t<1.
$$

因此，任何正的 Jacobian 有效速率都会预测一阶 loss 下降。

## 推论 2：多步收益在 log 空间中相加

如果未来优势跨步累计，

$$
\text{totalGain}=g_1+g_2+g_3,
$$

那么同一个证书变为

$$
\log(\text{prefixRatio})<g_1+g_2+g_3
\quad\Longrightarrow\quad
\text{prefixRatio}\exp(-(g_1+g_2+g_3))<1.
$$

这解释了为什么 notebook 比较累计 log 衰减，而不仅仅看一步 loss 下降。

## Lean4 验证

下面的代码已用 Lean 4.32.0 检查，只依赖 Lean core。它证明的是这些证书的 log 空间代数形式。连接 log 空间命题与指数公式的标准实分析步骤，已经在上面的数学文字中给出。

```lean
/-
Lean 4.32 core-verified log-domain tradeoff certificates.

The real-valued exponential predictor compares candidate b with baseline a:

  predicted_ratio = prefix_ratio * exp (- total_gain).

Taking logarithms gives the equivalent log-domain condition:

  log(predicted_ratio) = prefix_penalty - total_gain.

Thus predicted_ratio < 1 is certified by

  prefix_penalty < total_gain.

This file formalizes the log-domain algebra. The real-analysis facts about
log and exp are standard; using this log form avoids a heavy Mathlib cache
dependency while still machine-checking the decision rule used by the notebooks.
-/

def logPredictedRatio (prefixPenalty totalGain : Int) : Int :=
  prefixPenalty - totalGain

theorem log_tradeoff_certificate
    {prefixPenalty totalGain : Int}
    (h : prefixPenalty < totalGain) :
    logPredictedRatio prefixPenalty totalGain < 0 := by
  unfold logPredictedRatio
  exact Int.sub_neg_of_lt h

def accumulatedGain3 (g1 g2 g3 : Int) : Int :=
  g1 + g2 + g3

theorem accumulated_log_tradeoff_certificate
    {prefixPenalty g1 g2 g3 : Int}
    (h : prefixPenalty < accumulatedGain3 g1 g2 g3) :
    logPredictedRatio prefixPenalty (accumulatedGain3 g1 g2 g3) < 0 := by
  exact log_tradeoff_certificate h

def firstOrderLossRatio (rho : Int) : Int :=
  1 - 2 * rho

theorem positive_rate_improves_first_order_loss
    {rho : Int}
    (hrho : 0 < rho) :
    firstOrderLossRatio rho < 1 := by
  unfold firstOrderLossRatio
  omega

```

## 解读

这些定理是刻意克制的。它们不声称 Muon 或某个固定调度永远更好。它们证明的是 notebook 中使用的决策逻辑：

1. 正的 Jacobian 有效速率预测一阶下降。
2. 未来收缩优势可以抵消当前 loss 劣势。
3. 多步证据应该在 log 空间中累计。

图像检验的是这些假设在具体系统中是否足够准确；定理证明的是，在这些假设成立时，排序规则本身是数学上必然的。